In [1]:
import json
from pathlib import Path

import pandas as pd

from bcbench.results.base import BaseEvaluationResult, create_result_from_json

result_folder = Path("result")
runs: dict[str, list[BaseEvaluationResult]] = {}
for jsonl_file in result_folder.glob("*.jsonl"):
    runs[jsonl_file.stem] = [create_result_from_json(json.loads(line)) for line in jsonl_file.read_text(encoding="utf-8").splitlines()]

result_df = pd.DataFrame([{"run_id": run_id, **r.model_dump()} for run_id, results in runs.items() for r in results])

print(f"Category: {result_df['category'].iloc[0]}")
print(f"Agent: {result_df['agent_name'].iloc[0]}")
print(f"Model: {result_df['model'].iloc[0]}")
result_df = result_df.drop(columns=["category", "timeout", "error_message", "agent_name", "model", "experiment"])

print(f"Loaded {len(result_df)} results from {len(runs)} runs")
print(f"Unique instances: {result_df['instance_id'].nunique()}")
result_df.head(n=2)

Category: EvaluationCategory.BUG_FIX
Agent: GitHub Copilot CLI
Model: claude-haiku-4-5
Loaded 440 results from 8 runs
Unique instances: 55


,run_id,instance_id,project,resolved,build,generated_patch,metrics
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,diff --git a/src/Apps/W1/Shopify/App/src/Produ...,"{'execution_time': 110.779, 'llm_duration': 10..."
1,20104171950,microsoft__BCApps-4766,Subscription Billing,False,True,diff --git a/src/Apps/W1/Subscription Billing/...,"{'execution_time': 98.864, 'llm_duration': 94...."


In [2]:
from math import sqrt

import plotly.graph_objects as go

run_ids: list[str] = sorted(result_df["run_id"].unique())

cumulative_data = []
for i in range(2, len(run_ids) + 1):
    subset = result_df[result_df["run_id"].isin(run_ids[:i])]
    per_run = subset.groupby("run_id")["resolved"].mean() * 100

    sem = per_run.std(ddof=1) / sqrt(len(per_run))
    cumulative_data.append({"n_runs": i, "sem": sem})

sem_df = pd.DataFrame(cumulative_data)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=sem_df["n_runs"],
        y=sem_df["sem"],
        mode="lines+markers",
        name="SEM",
        line={"color": "blue", "width": 2},
        marker={"size": 10},
    )
)
fig.update_layout(
    title="SEM Convergence for % Resolved",
    xaxis_title="Number of Runs",
    yaxis_title="Standard Error of Mean (%)",
    xaxis={"tickmode": "linear", "tick0": 2, "dtick": 1},
    yaxis={"rangemode": "tozero"},
    template="plotly_white",
)
fig.show()

In [3]:
per_run = result_df.groupby("run_id")["resolved"].mean() * 100
mean: float = per_run.mean()
sem: float = per_run.std(ddof=1) / sqrt(len(per_run))

print(f"Mean resolution rate: {mean:.2f}%")
print(f"SEM: {sem:.2f}%")
print(f"95% CI: [{mean - 1.96 * sem:.2f}%, {mean + 1.96 * sem:.2f}%]")

Mean resolution rate: 45.45%
SEM: 1.46%
95% CI: [42.60%, 48.31%]


In [4]:
from bcbench.config import get_config
from bcbench.dataset import DatasetEntry, load_dataset_entries

_config = get_config()
bcbench_dataset: list[DatasetEntry] = load_dataset_entries(_config.paths.dataset_path)

dataset_df = pd.DataFrame(
    [
        {
            "instance_id": entry.instance_id,
            "image_count": entry.metadata.image_count or 0,
            "area": entry.metadata.area or "Unknown",
            "gold_patch": entry.patch,
        }
        for entry in bcbench_dataset
    ]
)

merged_df = result_df.merge(dataset_df, on="instance_id", how="left")
print(f"Merged {len(merged_df)} results with dataset metadata")
merged_df.head(n=2)

Merged 440 results with dataset metadata


,run_id,instance_id,project,resolved,build,generated_patch,metrics,image_count,area,gold_patch
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,diff --git a/src/Apps/W1/Shopify/App/src/Produ...,"{'execution_time': 110.779, 'llm_duration': 10...",5,shopify,diff --git a/src/Apps/W1/Shopify/App/src/Produ...
1,20104171950,microsoft__BCApps-4766,Subscription Billing,False,True,diff --git a/src/Apps/W1/Subscription Billing/...,"{'execution_time': 98.864, 'llm_duration': 94....",0,subscription billing,diff --git a/src/Apps/W1/Subscription Billing/...


In [5]:
from unidiff import PatchSet


def count_files_in_patch(patch: str) -> int:
    return len(PatchSet(patch))


merged_df["expected_files"] = merged_df["gold_patch"].apply(count_files_in_patch)

bins = [0, 1, 2, float("inf")]
labels = ["1", "2", "3+"]
merged_df["files_bin"] = pd.cut(merged_df["expected_files"], bins=bins, labels=labels)

instance_files_df = (
    merged_df.groupby("instance_id")
    .agg(
        files_bin=("files_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

files_resolution_df = (
    instance_files_df.groupby("files_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

total_instances = files_resolution_df["count"].sum()
files_resolution_df["% Resolved"] = (files_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
files_resolution_df["Instances"] = files_resolution_df["count"].astype(str) + " (" + (files_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
files_resolution_df = files_resolution_df.rename(columns={"files_bin": "Files Changed"})

print('Average "% Resolved" by Number of Files Changed (gold patch):')
print(files_resolution_df[["Files Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Number of Files Changed (gold patch):
Files Changed % Resolved  Instances
            1      51.6% 46 (83.6%)
            2       4.2%  6 (10.9%)
           3+      33.3%   3 (5.5%)


In [6]:
def count_loc_in_patch(patch: str) -> int:
    patch_set = PatchSet(patch)
    return patch_set.added + patch_set.removed


merged_df["expected_loc"] = merged_df["gold_patch"].apply(count_loc_in_patch)

bins = [0, 5, 10, 50, float("inf")]
labels = ["1-5", "6-10", "11-50", "50+"]
merged_df["loc_bin"] = pd.cut(merged_df["expected_loc"], bins=bins, labels=labels)

instance_loc_df = (
    merged_df.groupby("instance_id")
    .agg(
        loc_bin=("loc_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

loc_resolution_df = (
    instance_loc_df.groupby("loc_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

loc_resolution_df["% Resolved"] = (loc_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
loc_resolution_df["Instances"] = loc_resolution_df["count"].astype(str) + " (" + (loc_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
loc_resolution_df = loc_resolution_df.rename(columns={"loc_bin": "LoC Changed"})

print('Average "% Resolved" by Lines of Code Changed (gold patch):')
print(loc_resolution_df[["LoC Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Lines of Code Changed (gold patch):
LoC Changed % Resolved  Instances
        1-5      63.1% 22 (40.0%)
       6-10      47.9%  6 (10.9%)
      11-50      34.5% 21 (38.2%)
        50+      16.7%  6 (10.9%)


In [7]:
merged_df["project_group"] = merged_df["project"].apply(lambda x: "BaseApp" if x == "BaseApp" else "First-party Apps")

instance_project_df = (
    merged_df.groupby("instance_id")
    .agg(
        project_group=("project_group", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

project_resolution_df = (
    instance_project_df.groupby("project_group", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

project_resolution_df["% Resolved"] = (project_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
project_resolution_df["Instances"] = project_resolution_df["count"].astype(str) + " (" + (project_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
project_resolution_df = project_resolution_df.rename(columns={"project_group": "Project Group"})

print('Average "% Resolved" by Project Group:')
print(project_resolution_df[["Project Group", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Project Group:
   Project Group % Resolved  Instances
         BaseApp      44.2% 45 (81.8%)
First-party Apps      51.2% 10 (18.2%)


In [8]:
bins = [-1, 0, 5, 10, float("inf")]
labels = ["0", "1-5", "6-10", "10+"]
merged_df["image_bin"] = pd.cut(merged_df["image_count"], bins=bins, labels=labels)

instance_image_df = (
    merged_df.groupby("instance_id")
    .agg(
        image_bin=("image_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

image_resolution_df = (
    instance_image_df.groupby("image_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

image_resolution_df["% Resolved"] = (image_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
image_resolution_df["Instances"] = image_resolution_df["count"].astype(str) + " (" + (image_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
image_resolution_df = image_resolution_df.rename(columns={"image_bin": "Image Count"})

print('Average "% Resolved" by Image Count:')
print(image_resolution_df[["Image Count", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Image Count:
Image Count % Resolved  Instances
          0      47.1% 26 (47.3%)
        1-5      40.3%  9 (16.4%)
       6-10      44.2% 13 (23.6%)
        10+      48.2%  7 (12.7%)
